## Data Analytics

### Chapter 6: Data Aggregation and Transforming 

Data aggregation and transforming are essential processes in data analytics that help in organizing, summarizing, and modifying datasets to extract meaningful insights.

- **Grouping**: We can group data based on specific attributes or categories to analyze patterns and trends within subsets of the data. This allows for a more focused examination of the data.

- **Aggregation**: Combine multiple rows or columns into a single value based on a specified function (e.g., sum, mean, count). This is useful for summarizing data and extracting insights.

- **Transforming**: When using grouping, transforming allows us to apply functions to each group, enabling us to create new columns or modify existing ones based on the grouped data. This can help in deriving new metrics or features that are more relevant for analysis.

#### Generate Synthetic Data

In [500]:
import numpy as np
import pandas as pd

# Reproducible randomness
np.random.seed(42)

In [501]:
names = [
    "James", "Chris", "Adam", "Marian", "Peter",
    "Kevin", "Linda", "Emma", "Sophia", "Daniel",
    "Noah", "Olivia", "Lucas", "Grace", "Henry",
    "Liam", "Ava", "Mia", "Nathan", "Chloe"
]

majors = ["AI", "IT", "CSC", "SW"]

n = 20

df = pd.DataFrame({
    "Name": np.array(names),
    "Age": np.random.randint(18, 21, n),
    "Major": np.random.choice(majors, n),
    "GPA": np.round(np.random.uniform(2.0, 4.0, n), 2)
})

df

,Name,Age,Major,GPA
0,James,20,SW,2.91
1,Chris,18,SW,3.57
2,Adam,20,AI,2.40
3,Marian,20,AI,3.03
4,Peter,18,SW,3.18
5,Kevin,18,IT,2.09
6,Linda,20,IT,3.22
7,Emma,19,AI,2.34
8,Sophia,20,SW,2.13
9,Daniel,20,AI,3.90


#### 6.1. Basic Aggregation

Aggregation is a fundamental operation in data analysis that involves summarizing or combining data to extract meaningful insights. It allows us to compute summary statistics, such as sums, averages, counts, and other metrics, based on specific criteria or groupings.

In [502]:
# calculate the average GPA for each major
df.groupby("Major")["GPA"].mean()

Major
AI     3.120000
CSC    3.062500
IT     2.893333
SW     2.746250
Name: GPA, dtype: float64

In [503]:
# compute multiple statistics simultaneously
df.groupby("Major")["GPA"].agg(["mean", "median", "min", "max", "count"])

,mean,median,min,max,count
Major,,,,,
AI,3.120000,3.030,2.34,3.93,5
CSC,3.062500,3.115,2.20,3.82,4
IT,2.893333,3.220,2.09,3.37,3
SW,2.746250,2.895,2.07,3.57,8


#### 6.2. Advanced Aggregation

We can explore more advanced aggregation techniques by grouping data based on multiple attributes. This allows us to analyze patterns and trends within subsets of the data, providing a deeper understanding of the relationships between different variables.

In [504]:
# we can group students by both Major and Age
# the numeric_only=True parameter ensures that only numeric columns are included in the aggregation
df.groupby(["Major", "Age"]).mean(numeric_only=True)

GPA
Major Age          
AI    19   2.340000
      20   3.315000
CSC   18   2.610000
      19   3.820000
      20   2.910000
IT    18   2.090000
      19   3.370000
      20   3.220000
SW    18   3.210000
      19   2.433333
      20   2.520000

In [505]:
# You can also make them separate columns instead of a multi-index by using the unstack() method
df.groupby(["Major", "Age"]).mean(numeric_only=True).unstack()

GPA                 
Age      18        19     20
Major                       
AI      NaN  2.340000  3.315
CSC    2.61  3.820000  2.910
IT     2.09  3.370000  3.220
SW     3.21  2.433333  2.520

In [506]:
# specify different aggregation functions for different columns
df.groupby("Major").agg({
    "Age": "mean",
    "GPA": ["mean", "max", "min", "count"]
})

Age       GPA                  
         mean      mean   max   min count
Major                                    
AI     19.800  3.120000  3.93  2.34     5
CSC    19.250  3.062500  3.82  2.20     4
IT     19.000  2.893333  3.37  2.09     3
SW     18.875  2.746250  3.57  2.07     8

#### 6.3. Aggregation and Sorting

Sorting is an important step in data analysis that helps us organize and prioritize information based on specific criteria. By sorting aggregated data, we can identify trends, patterns, and outliers more effectively.

In [507]:
# rank majors by their average GPA
df.groupby("Major")["GPA"].mean().sort_values(ascending=True)

Major
SW     2.746250
IT     2.893333
CSC    3.062500
AI     3.120000
Name: GPA, dtype: float64

In [508]:
# sort using multiple aggregated columns
summary = df.groupby("Major").agg({
    "Age": "mean",
    "GPA": "mean"
})

summary.sort_values(
    by=["GPA", "Age"],
    ascending=[False, True]
)

,Age,GPA
Major,,
AI,19.800,3.120000
CSC,19.250,3.062500
IT,19.000,2.893333
SW,18.875,2.746250


#### 6.4. Aggregation with Lambda Functions

Lambda functions offer a concise way to define small, anonymous functions in Python. When combined with aggregation, lambda functions allow us to perform custom calculations on grouped data, enabling more flexible and tailored analyses.

In [509]:
# we want to calculate the GPA range (maximum − minimum) for each major
df.groupby("Major")["GPA"].agg(lambda x: x.max() - x.min())

Major
AI     1.59
CSC    1.62
IT     1.28
SW     1.50
Name: GPA, dtype: float64

In [510]:
# combine built-in functions with lambda functions
# rename the lambda function to "GPA Range" for clarity
df.groupby("Major")["GPA"].agg([
    "mean",
    "max",
    "min",
    ("GPA Range", lambda x: x.max() - x.min())
])

,mean,max,min,GPA Range
Major,,,,
AI,3.120000,3.93,2.34,1.59
CSC,3.062500,3.82,2.20,1.62
IT,2.893333,3.37,2.09,1.28
SW,2.746250,3.57,2.07,1.50


#### 6.5. Creating Derived Columns Using Group Information

The `transform` function in data analysis allows us to create new columns based on the information derived from grouped data. This is particularly useful for generating metrics or features that are relevant to specific groups, enhancing our ability to analyze and interpret the data effectively.

In [511]:
# Rename original GPA
df_new = df.copy()

df_new.rename(columns={"GPA": "GPA1"}, inplace=True)

# Generate another GPA
df_new["GPA2"] = np.round(np.random.uniform(2.0, 4.0, len(df_new)), 2)
# Final GPA
df_new["Final_GPA"] = (df_new["GPA1"] + df_new["GPA2"]) / 2

df_new

,Name,Age,Major,GPA1,GPA2,Final_GPA
0,James,20,SW,2.91,2.52,2.715
1,Chris,18,SW,3.57,3.33,3.450
2,Adam,20,AI,2.40,2.62,2.510
3,Marian,20,AI,3.03,3.04,3.035
4,Peter,18,SW,3.18,3.09,3.135
5,Kevin,18,IT,2.09,2.37,2.230
6,Linda,20,IT,3.22,3.94,3.580
7,Emma,19,AI,2.34,3.55,2.945
8,Sophia,20,SW,2.13,3.88,3.005
9,Daniel,20,AI,3.90,3.79,3.845


In [512]:
# Creating a Scholarship Column
def scholarship(row):
    if row["Major"] in ["SW", "CSC"] and row["Final_GPA"] >= 3.2:
        return "Web Engineering"
    elif row["Major"] in ["AI", "IT"] and row["Final_GPA"] >= 3.2:
        return "Data Science"
    else:
        return "None"

df_new["Scholarship"] = df_new.apply(scholarship, axis=1)

df_new[["Name", "Major", "Final_GPA", "Scholarship"]][df_new["Scholarship"] != "None"]

,Name,Major,Final_GPA,Scholarship
1,Chris,SW,3.450,Web Engineering
6,Linda,IT,3.580,Data Science
9,Daniel,AI,3.845,Data Science
10,Noah,AI,3.565,Data Science
11,Olivia,CSC,3.730,Web Engineering
19,Chloe,CSC,3.265,Web Engineering


In [513]:
# Creating Derived Columns with transform()
# First, compute the average GPA for each major and assign it back to every student in that major.
df_new["Major_Avg_GPA"] = df_new.groupby("Major")["Final_GPA"].transform("mean")
df_new[["Major", "Major_Avg_GPA"]]

,Major,Major_Avg_GPA
0,SW,2.901250
1,SW,2.901250
2,AI,3.180000
3,AI,3.180000
4,SW,2.901250
5,IT,2.846667
6,IT,2.846667
7,AI,3.180000
8,SW,2.901250
9,AI,3.180000


In [514]:
# Then create a new column indicating whether each student is above the group average.
df_new["Above_Average"] = (df_new["Final_GPA"] > df_new["Major_Avg_GPA"])
df_new[["Final_GPA", "Major_Avg_GPA", "Above_Average"]]

,Final_GPA,Major_Avg_GPA,Above_Average
0,2.715,2.901250,False
1,3.450,2.901250,True
2,2.510,3.180000,False
3,3.035,3.180000,False
4,3.135,2.901250,True
5,2.230,2.846667,False
6,3.580,2.846667,True
7,2.945,3.180000,False
8,3.005,2.901250,True
9,3.845,3.180000,True


#### 6.6. Imputing Missing Values with Group Statistics

Imputing missing values is a common task in data preprocessing, and using group statistics can provide more accurate estimates. 

By leveraging the information from specific groups, we can fill in missing values with group-specific metrics, such as the mean or median, rather than relying on global statistics.

In [515]:
# 25% missing values 
df_missing = df_new.copy()
df_missing.loc[df_missing.sample(frac=0.25).index, "Final_GPA"] = np.nan

df_missing[['Name', 'Major', 'Final_GPA']]

,Name,Major,Final_GPA
0,James,SW,2.715
1,Chris,SW,NaN
2,Adam,AI,2.510
3,Marian,AI,NaN
4,Peter,SW,NaN
5,Kevin,IT,NaN
6,Linda,IT,3.580
7,Emma,AI,2.945
8,Sophia,SW,3.005
9,Daniel,AI,3.845


In [516]:
# Imputation using the global mean
df_imputed1 = df_missing.copy()
df_imputed1["Final_GPA"] = df_imputed1["Final_GPA"].fillna(df_imputed1["Final_GPA"].mean())

df_imputed1[['Name', 'Major', 'Final_GPA']]

,Name,Major,Final_GPA
0,James,SW,2.715
1,Chris,SW,2.928
2,Adam,AI,2.510
3,Marian,AI,2.928
4,Peter,SW,2.928
5,Kevin,IT,2.928
6,Linda,IT,3.580
7,Emma,AI,2.945
8,Sophia,SW,3.005
9,Daniel,AI,3.845


In [517]:
# Instead of using the global mean, we can fill missing values 
# using each major’s average GPA.
df_imputed2 = df_missing.copy()
df_imputed2["Final_GPA"] = df_imputed2["Final_GPA"].fillna(
			df_imputed2.groupby("Major")["Final_GPA"].transform("mean"))

df_imputed2[['Name', 'Major', 'Final_GPA']]

,Name,Major,Final_GPA
0,James,SW,2.715000
1,Chris,SW,2.770833
2,Adam,AI,2.510000
3,Marian,AI,3.100000
4,Peter,SW,2.770833
5,Kevin,IT,3.155000
6,Linda,IT,3.580000
7,Emma,AI,2.945000
8,Sophia,SW,3.005000
9,Daniel,AI,3.845000


#### 6.7. Other Common Transformations

You can utilize the `transform` function to apply various transformations to your data, such as scaling, normalization, or custom calculations. This allows for more advanced data manipulation and feature engineering, enabling you to prepare your dataset for further analysis or modeling.

In [518]:
# Finding Each Student’s Contribution to Their Group
df_new1 = df_new.copy()

df_new1["Contribution (%)"] = (df_new1["Final_GPA"] / 
                      df_new1.groupby("Major")["Final_GPA"].transform("sum")) 

df_new1["Contribution (%)"] = np.round(df_new1["Contribution (%)"] * 100, 2)

df_new1[['Name', 'Major', 'Final_GPA', 'Contribution (%)']]

,Name,Major,Final_GPA,Contribution (%)
0,James,SW,2.715,11.70
1,Chris,SW,3.450,14.86
2,Adam,AI,2.510,15.79
3,Marian,AI,3.035,19.09
4,Peter,SW,3.135,13.51
5,Kevin,IT,2.230,26.11
6,Linda,IT,3.580,41.92
7,Emma,AI,2.945,18.52
8,Sophia,SW,3.005,12.95
9,Daniel,AI,3.845,24.18


In [519]:
# Optionally, filter out the AI major
df_new1[['Name', 'Major', 'Final_GPA', 
         'Contribution (%)']][df_new1['Major'] == 'AI']

,Name,Major,Final_GPA,Contribution (%)
2,Adam,AI,2.510,15.79
3,Marian,AI,3.035,19.09
7,Emma,AI,2.945,18.52
9,Daniel,AI,3.845,24.18
10,Noah,AI,3.565,22.42


In [520]:
# Group-wise Standardization (Z-Score)
df_new1["Z_Score"] = (df_new1.groupby("Major")["Final_GPA"]
								.transform(lambda x: (x - x.mean()) / x.std()))

df_new1[['Name', 'Major', 'Final_GPA', 'Z_Score']]

,Name,Major,Final_GPA,Z_Score
0,James,SW,2.715,-0.640050
1,Chris,SW,3.450,1.885786
2,Adam,AI,2.510,-1.268675
3,Marian,AI,3.035,-0.274564
4,Peter,SW,3.135,0.803285
5,Kevin,IT,2.230,-0.903515
6,Linda,IT,3.580,1.074450
7,Emma,AI,2.945,-0.444983
8,Sophia,SW,3.005,0.356538
9,Daniel,AI,3.845,1.259208


In [521]:
# Optionally, filter out the AI major
df_new1[['Name', 'Major', 'Final_GPA', 
         'Z_Score']][df_new1['Major'] == 'AI']

,Name,Major,Final_GPA,Z_Score
2,Adam,AI,2.510,-1.268675
3,Marian,AI,3.035,-0.274564
7,Emma,AI,2.945,-0.444983
9,Daniel,AI,3.845,1.259208
10,Noah,AI,3.565,0.729015
